# **Undersampling Cluster Centroids**

In [ ]:
# !pip install pyspark
from pyspark.sql import SparkSession
ss = SparkSession.builder \
    .master("local[*]") \
    .appName("imbalanced") \
    .getOrCreate()


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
path_train = "/content/drive/MyDrive/DDAM project/data/train_test/train_set.parquet"
path_test  = "/content/drive/MyDrive/DDAM project/data/train_test/test_set.parquet"

df_train = ss.read.parquet(path_train)
df_test = ss.read.parquet(path_test)

In [ ]:
df_train.show()

In [ ]:
df_train.dtypes

## Drop columns

In [ ]:
columns_to_drop = ["XCoords", "YCoords", "Image", "Tile", "Date"]

df_train = df_train.drop(*columns_to_drop)
df_test = df_test.drop(*columns_to_drop)

In [ ]:
df_train.show()

## Confidence Conversion

In [ ]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder

# Converts Confidence (string) to a numeric index
confidence_indexer = StringIndexer(
    inputCol="Confidence",
    outputCol="Confidence_indexed",
    handleInvalid="keep"
)

# One-Hot Encoding of the Confidence index
confidence_ohe = OneHotEncoder(
    inputCol="Confidence_indexed",
    outputCol="Confidence_ohe",
    dropLast=False  # Retains all categories
)

# **ClusterCentroids**

In [ ]:
import numpy as np
from functools import reduce
from pyspark.sql import DataFrame
from pyspark.sql.functions import col, udf
from pyspark.sql.types import StructType, StructField, DoubleType, StringType
from pyspark.ml.linalg import Vectors, VectorUDT
from pyspark.ml.clustering import KMeans
from pyspark.ml.feature import StandardScaler, StringIndexer


class ClusterCentroids:
    """
    PySpark implementation of Cluster Centroids (undersampling).

    """

    def __init__(
        self,
        label_col: str = "label",
        features_col: str = "features",
        target: int = None,
        sampling_ratios: dict = None,
        scale_features: bool = True,
        kmeans_max_iter: int = 20,
        seed: int = 42
    ):
        self.label_col = label_col
        self.features_col = features_col
        self.target = target
        self.sampling_ratios = sampling_ratios or {}
        self.scale_features = scale_features
        self.kmeans_max_iter = kmeans_max_iter
        self.seed = seed

    # ------------------------------------------------------------------ #
    #  Utility: calculate k target for each class                       #
    # ------------------------------------------------------------------ #
    def _target_k(self, label, class_count: int, minority_count: int) -> int:
        if label in self.sampling_ratios:
            k = max(1, int(class_count * self.sampling_ratios[label]))
        elif self.target is not None:
            k = self.target          # use the absolute threshold
        else:
            k = minority_count
        return min(k, class_count)   # never > count, avoids the KMeans bug

    # ------------------------------------------------------------------ #
    #  Core: undersample one class with K-means                            #
    # ------------------------------------------------------------------ #
    def _undersample(self, class_df: DataFrame, k: int, label_val: float,
                     scaler_model=None) -> DataFrame:
        spark = class_df.sparkSession

        feat_col = self.features_col
        if scaler_model is not None:
            class_df = scaler_model.transform(class_df)
            feat_col = "_features_scaled"

        kmeans = KMeans(k=k, seed=self.seed, featuresCol=feat_col,
                        maxIter=self.kmeans_max_iter)
        model = kmeans.fit(class_df)
        centers = model.clusterCenters()

        # Invert scaling if applied
        if scaler_model is not None:
            mean_vals = scaler_model.mean.toArray()
            std_vals = scaler_model.std.toArray()
            centers = [c * std_vals + mean_vals for c in centers]

        schema = StructType([
            StructField(self.features_col, VectorUDT(), False),
            StructField(self.label_col, DoubleType(), False)
        ])
        center_rows = [(Vectors.dense(c.tolist()), float(label_val)) for c in centers]
        return spark.createDataFrame(center_rows, schema)

    # ------------------------------------------------------------------ #
    #  fit_transform: equivalent of ADASYN.fit_transform                 #
    # ------------------------------------------------------------------ #
    def fit_transform(self, df: DataFrame) -> DataFrame:
        spark = df.sparkSession

        # robust schema access (avoid PySparkKeyError)
        if self.label_col not in df.columns:
            raise ValueError(
                f"Colonna '{self.label_col}' non trovata. "
                f"Colonne disponibili: {df.columns}"
            )

        label_field = next(f for f in df.schema.fields if f.name == self.label_col)
        label_is_string = isinstance(label_field.dataType, StringType)

        if label_is_string:
            indexer = StringIndexer(inputCol=self.label_col, outputCol="_label_idx")
            indexer_model = indexer.fit(df)
            df = indexer_model.transform(df) \
                .drop(self.label_col) \
                .withColumnRenamed("_label_idx", self.label_col)
            labels_list = indexer_model.labels
        else:
            labels_list = None

        df = df.select(
            col(self.label_col).cast(DoubleType()).alias(self.label_col),
            col(self.features_col)
        )

        # count for class
        counts_rows = df.groupBy(self.label_col).count().orderBy(col("count").asc()).collect()
        counts = {row[self.label_col]: row["count"] for row in counts_rows}
        minority_count = counts_rows[0]["count"]  # smallest class

        print(f"Distribuzione classi input: { {k: v for k, v in counts.items()} }")
        print(f"Target k (default): {minority_count}")

        # StandardScaler global (optional)
        scaler_model = None
        if self.scale_features:
            scaler = StandardScaler(inputCol=self.features_col, outputCol="_features_scaled",
                                    withMean=True, withStd=True)
            scaler_model = scaler.fit(df)

        # Cluster Centroids per class
        resampled = []
        for label_val, count in counts.items():
            class_df = df.filter(col(self.label_col) == label_val)
            k = self._target_k(label_val, count, minority_count)
            k = min(k, count)

            if count <= k:
                # under or equal to the target: remain unchanged
                resampled.append(class_df.select(self.features_col, self.label_col))
                print(f"Classe {label_val}: {count} campioni → invariata")
            else:
                centers_df = self._undersample(class_df, k, label_val, scaler_model)
                resampled.append(centers_df)
                print(f"Classe {label_val}: {count} campioni → {k} centroidi")

        result = reduce(lambda a, b: a.union(b), resampled)

        # restore string label if necessary
        if label_is_string and labels_list is not None:
            idx_to_str = udf(lambda i: labels_list[int(i)], StringType())
            result = result \
                .withColumn("_original_label", idx_to_str(col(self.label_col).cast("int"))) \
                .drop(self.label_col) \
                .withColumnRenamed("_original_label", self.label_col)

        return result


In [ ]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler
from pyspark.ml import Pipeline as PrePipeline


spark = SparkSession.builder.appName("Cluster Centroids").getOrCreate()


# Fit a lightweight preprocessing pipeline to encode the Confidence feature
preprocessing_pipeline = PrePipeline(stages=[confidence_indexer, confidence_ohe])
preprocessing_model = preprocessing_pipeline.fit(df_train)


# Apply preprocessing to df_train, adding the Confidence_ohe column
df_train = preprocessing_model.transform(df_train)


feature_cols = ["NDVI", "FDI", "NDWI", "NRD", "NDMI", "BSI", "CON", "DIS", "ENER", "COR", "PC_1", "Confidence_ohe"]


assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
df_assembled = assembler.transform(df_train).select("Class", "features")


# Undersample majority classes using cluster centroids, targeting 10k samples per class
cc = ClusterCentroids(
    label_col="Class",
    features_col="features",
    target=10000,
    scale_features=False,
    seed=42
)


df_train = cc.fit_transform(df_assembled)
df_train.groupBy("Class").count().show()


In [ ]:
from pyspark.ml.functions import vector_to_array


# Same feature names and order as defined in the VectorAssembler
feature_names = ["NDVI", "FDI", "NDWI", "NRD", "NDMI", "BSI",
                 "CON", "DIS", "ENER", "COR", "PC_1"]


# Append OHE column names for Confidence (one per category, e.g. 3: High/Medium/Low)
n_confidence_cats = 3  # adjust to the actual number of categories
ohe_names = [f"Confidence_ohe_{i}" for i in range(n_confidence_cats)]


all_feature_names = feature_names + ohe_names


# Explode the features vector into individual columns
df_to_save = df_train.withColumn("_arr", vector_to_array(col("features")))


for i, name in enumerate(all_feature_names):
    df_to_save = df_to_save.withColumn(name, col("_arr")[i])


# Drop intermediate vector columns before saving
df_to_save = df_to_save.drop("features", "_arr")


df_to_save.write.csv(
    "/content/drive/MyDrive/DDAM project/data/df_train_UNDERSAMPLING.csv",
    header=True,
    mode="overwrite"
)


## Random search Cross Validation



In [ ]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml import Pipeline
from xgboost.spark import SparkXGBClassifier

feature_cols = [
    "NDVI", "FDI", "NDWI", "NRD", "NDMI", "BSI", "CON", "DIS", "ENER", "COR", "PC_1", "Confidence_ohe"
]

label = "Class"

# Indexes the target column into numerical labels
indexer = StringIndexer(inputCol=label, outputCol="label")

# Assembles the features into a single column vector
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")

# Defines the XGBoost classifier
xgb_estimator = SparkXGBClassifier(
    features_col="features",
    label_col="label",
    num_workers=2,
    seed=42
)

In [ ]:
import random
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1"
)

pipeline = Pipeline(stages=[confidence_indexer, confidence_ohe, indexer, assembler, xgb_estimator])

# Defines a grid larger than NUM_SAMPLES to enable random sampling
large_param_grid = (
    ParamGridBuilder()
    .addGrid(xgb_estimator.max_depth, [4, 6, 8])        # 3 options
    .addGrid(xgb_estimator.subsample, [0.7, 0.8, 1.0])  # 3 options
    .addGrid(xgb_estimator.n_estimators, [50, 100])     # 2 options
    .build()  # Total: 3 * 3 * 2 = 18 combinations
)

NUM_SAMPLES = 8
SEED = 42

# Randomly samples NUM_SAMPLES combinations from the full grid
sampled_paramGrid = random.sample(large_param_grid, NUM_SAMPLES)

crossval = CrossValidator(
    estimator=pipeline,
    estimatorParamMaps=sampled_paramGrid,
    evaluator=evaluator,
    numFolds=5,
    seed=SEED
)

print(f"\n Avvio Random Search Cross-Validation ({NUM_SAMPLES} combinazioni)...")

cvModel = crossval.fit(df_train)

# Extract the best model and the average CV metrics
bestModel = cvModel.bestModel
avgMetrics = cvModel.avgMetrics

print(f"\n Cross-Validation completata. Best F1: {max(avgMetrics):.4f}")

In [ ]:
OUTPUT_PATH = "/content/drive/MyDrive/DDAM project/models/best_xgb_model_UNDERSAMPLING"

# Save on Drive
print(f"Salvataggio del modello in corso su: {OUTPUT_PATH} ...")
bestModel.write().overwrite().save(OUTPUT_PATH)

print("Modello salvato con successo!")


In [ ]:
from pyspark.sql.functions import col


# Apply the same preprocessing pipeline used before undersampling
df_test_pre = preprocessing_model.transform(df_test)  # adds Confidence_indexed, Confidence_ohe


# Assemble features using the same columns as in training
feature_cols = ["NDVI", "FDI", "NDWI", "NRD", "NDMI", "BSI",
                "CON", "DIS", "ENER", "COR", "PC_1", "Confidence_ohe"]
assembler_test = VectorAssembler(inputCols=feature_cols, outputCol="features")


df_test = assembler_test.transform(df_test_pre).select("features", "Class")


# Apply the best model to the test set
predictions = bestModel.transform(df_test)
predictions.cache()
predictions.count()  # Forces materialization (cache is lazy)


predictions.select(col("label"), col("prediction")).show(5)


metrics_to_evaluate = ["accuracy", "f1", "weightedPrecision", "weightedRecall"]


print("\n--- Detailed Performance Metrics (Test Set) ---")


results = {}
for metric_name in metrics_to_evaluate:
    test_evaluator = MulticlassClassificationEvaluator(
        labelCol="label",
        predictionCol="prediction",
        metricName=metric_name
    )
    metric_value = test_evaluator.evaluate(predictions)
    results[metric_name] = metric_value
    print(f" {metric_name.ljust(20)}: {metric_value:.4f}")


# Confusion matrix — essential for per-class precision/recall analysis
print("\n--- Confusion Matrix ---")


confusion_matrix = predictions.groupBy('label').pivot('prediction').count().fillna(0).orderBy('label')
confusion_matrix.show()


In [ ]:
from pyspark.ml.feature import IndexToString
from pyspark.mllib.evaluation import MulticlassMetrics


# Converts the numerical prediction indices back to the original string labels
labelConverter = IndexToString(
    inputCol="prediction",
    outputCol="predictedLabel",
    labels=bestModel.stages[2].labels
)


predictions_with_strings = labelConverter.transform(predictions)
predictions_with_strings.cache()
predictions_with_strings.count()


# Actual count per class
support_counts_df = predictions_with_strings.groupBy("label").count()
support_map = support_counts_df.rdd.collectAsMap()


# MulticlassMetrics requires an RDD of (prediction, label) pairs as Double
prediction_and_labels = (
    predictions_with_strings.select(col("prediction"), col("label"))
    .rdd.map(lambda row: tuple(map(float, row)))
)


metrics = MulticlassMetrics(prediction_and_labels)
labels = predictions_with_strings.select("label").distinct().rdd.flatMap(lambda x: x).collect()


print(f"{'Classe':<25} {'Support':<10} {'Precision':<15} {'Recall':<15} {'F1-Score':<15}")
print("-" * 80)


for label_index in sorted(labels):
    class_precision = metrics.precision(label=label_index)
    class_recall    = metrics.recall(label=label_index)
    class_f1        = metrics.fMeasure(label=label_index)
    class_support   = support_map.get(label_index, 0)
    class_string    = bestModel.stages[2].labels[int(label_index)]


    print(f"{class_string:<25.25} {class_support:<10} {class_precision:<15.4f} {class_recall:<15.4f} {class_f1:<15.4f}")


print("-" * 80)


predictions_with_strings.unpersist()
predictions.unpersist()


In [ ]:
# ── SAVE ClusterCentroids Multi ───────────────────────────────────────────────
_labels_cc_multi = bestModel.stages[0].labels
_md_idx_cc = float(_labels_cc_multi.index("Marine Debris"))
saved_cc_multi = {
    "config":    "ClusterC. Multi",
    "precision": round(metrics.precision(label=_md_idx_cc), 4),
    "recall":    round(metrics.recall(label=_md_idx_cc),    4),
    "f1":        round(metrics.fMeasure(label=_md_idx_cc),  4),
}
print("Saved ClusterC. Multi:", saved_cc_multi)


# **Undersampling Variabile Target Binaria**

In [ ]:
# !pip install pyspark
from pyspark.sql import SparkSession
ss = SparkSession.builder \
    .master("local[*]") \
    .appName("imbalanced") \
    .getOrCreate()


In [ ]:
path_train = "/content/drive/MyDrive/DDAM project/data/train_test/train_set.parquet"
path_test  = "/content/drive/MyDrive/DDAM project/data/train_test/test_set.parquet"

df_train = ss.read.parquet(path_train)
df_test = ss.read.parquet(path_test)

In [ ]:
df_train.show()

In [ ]:
df_train.dtypes

In [ ]:
from pyspark.sql import functions as F

# Marine Debris = 1, Others = 0
df_train = df_train.withColumn(
  'Class',
  F.when(F.col('Class') == 'Marine Debris', 1).otherwise(0)
)

df_test = df_test.withColumn(
  'Class',
  F.when(F.col('Class') == 'Marine Debris', 1).otherwise(0)
)

## Drop columns

In [ ]:
columns_to_drop = ["XCoords", "YCoords", "Image", "Tile", "Date"]

df_train = df_train.drop(*columns_to_drop)
df_test = df_test.drop(*columns_to_drop)

In [ ]:
df_train.show()

## Confidence Conversion

In [ ]:
# Converts Confidence (string) to a numeric index
confidence_indexer = StringIndexer(
    inputCol="Confidence",
    outputCol="Confidence_indexed",
    handleInvalid="keep"
)

# One-Hot Encoding of the Confidence index
confidence_ohe = OneHotEncoder(
    inputCol="Confidence_indexed",
    outputCol="Confidence_ohe",
    dropLast=False  # Retains all categories
)

# **ClusterCentroids**

In [ ]:
spark = SparkSession.builder.appName("Cluster Centroids").getOrCreate()


# Fit a lightweight preprocessing pipeline to encode the Confidence feature
preprocessing_pipeline = PrePipeline(stages=[confidence_indexer, confidence_ohe])
preprocessing_model = preprocessing_pipeline.fit(df_train)


# Apply preprocessing to df_train, adding the Confidence_ohe column
df_train = preprocessing_model.transform(df_train)


feature_cols = ["NDVI", "FDI", "NDWI", "NRD", "NDMI", "BSI", "CON", "DIS", "ENER", "COR", "PC_1", "Confidence_ohe"]


assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
df_assembled = assembler.transform(df_train).select("Class", "features")


# Undersample majority classes using cluster centroids, targeting 10k samples per class
cc = ClusterCentroids(
    label_col="Class",
    features_col="features",
    target=10000,
    scale_features=False,
    seed=42
)


df_train = cc.fit_transform(df_assembled)
df_train.groupBy("Class").count().show()


In [ ]:

# Same feature names and order as defined in the VectorAssembler
feature_names = ["NDVI", "FDI", "NDWI", "NRD", "NDMI", "BSI",
                 "CON", "DIS", "ENER", "COR", "PC_1"]


# Append OHE column names for Confidence (one per category, e.g. 3: High/Medium/Low)
n_confidence_cats = 3  # adjust to the actual number of categories
ohe_names = [f"Confidence_ohe_{i}" for i in range(n_confidence_cats)]


all_feature_names = feature_names + ohe_names


# Explode the features vector into individual columns
df_to_save = df_train.withColumn("_arr", vector_to_array(col("features")))


for i, name in enumerate(all_feature_names):
    df_to_save = df_to_save.withColumn(name, col("_arr")[i])


# Drop intermediate vector columns before saving
df_to_save = df_to_save.drop("features", "_arr")


df_to_save.write.csv(
    "/content/drive/MyDrive/DDAM project/data/df_train_binary_UNDERSAMPLING.csv",
    header=True,
    mode="overwrite"
)


## Random search Cross Validation



In [ ]:
feature_cols = [
    "NDVI", "FDI", "NDWI", "NRD", "NDMI", "BSI", "CON", "DIS", "ENER", "COR", "PC_1", "Confidence_ohe"
]

label = "Class"

# Indexes the target column into numerical labels
indexer = StringIndexer(inputCol=label, outputCol="label")

# Assembles the features into a single column vector
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")

# Defines the XGBoost classifier
xgb_estimator = SparkXGBClassifier(
    features_col="features",
    label_col="label",
    num_workers=2,
    seed=42
)

In [ ]:
evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1"
)

pipeline = Pipeline(stages=[confidence_indexer, confidence_ohe, indexer, assembler, xgb_estimator])

# Defines a grid larger than NUM_SAMPLES to enable random sampling
large_param_grid = (
    ParamGridBuilder()
    .addGrid(xgb_estimator.max_depth, [4, 6, 8])        # 3 options
    .addGrid(xgb_estimator.subsample, [0.7, 0.8, 1.0])  # 3 options
    .addGrid(xgb_estimator.n_estimators, [50, 100])     # 2 options
    .build()  # Total: 3 * 3 * 2 = 18 combinations
)

NUM_SAMPLES = 8
SEED = 42

# Randomly samples NUM_SAMPLES combinations from the full grid
sampled_paramGrid = random.sample(large_param_grid, NUM_SAMPLES)

crossval = CrossValidator(
    estimator=pipeline,
    estimatorParamMaps=sampled_paramGrid,
    evaluator=evaluator,
    numFolds=5,
    seed=SEED
)

print(f"\n Avvio Random Search Cross-Validation ({NUM_SAMPLES} combinazioni)...")

cvModel = crossval.fit(df_train)

# Extract the best model and the average CV metrics
bestModel = cvModel.bestModel
avgMetrics = cvModel.avgMetrics

print(f"\n Cross-Validation completata. Best F1: {max(avgMetrics):.4f}")

In [ ]:
OUTPUT_PATH = "/content/drive/MyDrive/DDAM project/models/best_xgb_model_UNDERSAMPLING"

# Save on Drive
print(f"Salvataggio del modello in corso su: {OUTPUT_PATH} ...")
bestModel.write().overwrite().save(OUTPUT_PATH)

print("Modello salvato con successo!")


In [ ]:
from pyspark.sql.functions import col
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator

# --- Preprocessing & Feature Assembly ---
feature_cols = ["NDVI", "FDI", "NDWI", "NRD", "NDMI", "BSI",
                "CON", "DIS", "ENER", "COR", "PC_1", "Confidence_ohe"]

df_test_pre = preprocessing_model.transform(df_test)
df_test_assembled = VectorAssembler(inputCols=feature_cols, outputCol="features") \
                        .transform(df_test_pre) \
                        .select("features", col("Class").cast("double"))  # ← FIX: allinea il tipo a quello del train
# --- Predizioni ---
predictions = bestModel.transform(df_test_assembled).cache()
predictions.count()  # materializza cache

# --- Metriche ---
for metric in ["accuracy", "f1", "weightedPrecision", "weightedRecall"]:
    val = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName=metric).evaluate(predictions)
    print(f"{metric:<20}: {val:.4f}")

auc = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC").evaluate(predictions)
print(f"{'AUC-ROC':<20}: {auc:.4f}")

# --- Confusion Matrix ---
print("\n--- Confusion Matrix ---")
predictions.groupBy("label").pivot("prediction").count().fillna(0).orderBy("label").show()


In [ ]:
# Converts the numerical prediction indices back to the original string labels
labelConverter = IndexToString(
    inputCol="prediction",
    outputCol="predictedLabel",
    labels=bestModel.stages[2].labels
)


predictions_with_strings = labelConverter.transform(predictions)
predictions_with_strings.cache()
predictions_with_strings.count()


# Actual count per class -> dictionary for fast lookup in the loop
support_counts_df = predictions_with_strings.groupBy("label").count()
support_map = support_counts_df.rdd.collectAsMap()


# MulticlassMetrics requires an RDD of (prediction, label) pairs as Double
prediction_and_labels = (
    predictions_with_strings.select(col("prediction"), col("label"))
    .rdd.map(lambda row: tuple(map(float, row)))
)


metrics = MulticlassMetrics(prediction_and_labels)
labels = predictions_with_strings.select("label").distinct().rdd.flatMap(lambda x: x).collect()


print("\n--- Per-Class Metrics (Test Set) ---")
print(f"{'Class':<25} {'Support':<10} {'Precision':<15} {'Recall':<15} {'F1-Score':<15}")
print("-" * 80)


for label_index in sorted(labels):
    class_precision = metrics.precision(label=label_index)
    class_recall    = metrics.recall(label=label_index)
    class_f1        = metrics.fMeasure(label=label_index)
    class_support   = support_map.get(label_index, 0)
    class_string    = bestModel.stages[2].labels[int(label_index)]


    print(f"{class_string:<25.25} {class_support:<10} {class_precision:<15.4f} {class_recall:<15.4f} {class_f1:<15.4f}")


print("-" * 80)


predictions_with_strings.unpersist()
predictions.unpersist()


In [ ]:
# ── SAVE ClusterCentroids Binary ──────────────────────────────────────────────
# Marine Debris = label 1.0 nel caso binario
saved_cc_binary = {
    "config":    "ClusterC. Binary",
    "precision": round(metrics.precision(label=1.0), 4),
    "recall":    round(metrics.recall(label=1.0),    4),
    "f1":        round(metrics.fMeasure(label=1.0),  4),
}
print("Saved ClusterC. Binary:", saved_cc_binary)


 # **Oversampling ADASYN**

In [ ]:
# !pip install pyspark
ss = SparkSession.builder \
    .master("local[*]") \
    .appName("imbalanced") \
    .getOrCreate()


In [ ]:
path_train = "/content/drive/MyDrive/DDAM project/data/train_test/train_set.parquet"
path_test  = "/content/drive/MyDrive/DDAM project/data/train_test/test_set.parquet"

df_train = ss.read.parquet(path_train)
df_test = ss.read.parquet(path_test)

In [ ]:
df_train.show()

In [ ]:
df_train.dtypes

## Drop columns

In [ ]:
columns_to_drop = ["XCoords", "YCoords", "Image", "Tile", "Date"]

df_train = df_train.drop(*columns_to_drop)
df_test = df_test.drop(*columns_to_drop)

In [ ]:
df_train.show()

## Confidence Conversion

In [ ]:
# Converts Confidence (string) to a numeric index
confidence_indexer = StringIndexer(
    inputCol="Confidence",
    outputCol="Confidence_indexed",
    handleInvalid="keep"
)

# One-Hot Encoding of the Confidence index
confidence_ohe = OneHotEncoder(
    inputCol="Confidence_indexed",
    outputCol="Confidence_ohe",
    dropLast=False  # Retains all categories
)

# **ADASYN**

In [ ]:
import numpy as np
import random
from sklearn.neighbors import NearestNeighbors
from pyspark.sql import DataFrame
from pyspark.sql.functions import col, desc, udf
from pyspark.sql.types import StructType, StructField, DoubleType, StringType
from pyspark.ml.linalg import Vectors, VectorUDT
from pyspark.ml.feature import StringIndexer



class ADASYN:
    """
    PySpark implementation of ADASYN (Adaptive Synthetic Sampling).
    Faithful to the Scala implementation by fsleeman/spark-class-balancing.

    Uses sklearn NearestNeighbors instead of spark-knn.
    """


    def __init__(
        self,
        k: int = 5,
        label_col: str = "label",
        features_col: str = "features",
        target: int = None,
        sampling_ratios: dict = None,
        oversamples_only: bool = False
    ):
        self.k = k
        self.label_col = label_col
        self.features_col = features_col
        self.target = target
        self.sampling_ratios = sampling_ratios or {}
        self.oversamples_only = oversamples_only


    # ------------------------------------------------------------------ #
    #  Utility: equivalent of createSmoteStyleExample in Utilities.scala  #
    # ------------------------------------------------------------------ #
    @staticmethod
    def _interpolate(x: np.ndarray, neighbor: np.ndarray) -> np.ndarray:
        """Creates a synthetic example by interpolating x and neighbor."""
        gap = random.random()
        return x + gap * (neighbor - x)


    # ------------------------------------------------------------------ #
    #  Utility: computes how many samples to add for each class           #
    # ------------------------------------------------------------------ #
    def _samples_to_add(self, label, class_count: int, majority_count: int) -> int:
      if label in self.sampling_ratios:
          return max(0, int(class_count * self.sampling_ratios[label]) - class_count)
      elif self.target is not None:
          # add only if below target, otherwise 0
          return max(0, self.target - class_count)
      return max(0, majority_count - class_count)


    # ------------------------------------------------------------------ #
    #  Equivalent of oversample() in ADASYN.scala                        #
    # ------------------------------------------------------------------ #
    def _oversample(self, all_features, all_labels, minority_label, samples_to_add):
      if samples_to_add <= 0:
          return []


      minority_mask = (all_labels == minority_label)
      minority_features = all_features[minority_mask]


      if len(minority_features) < 2:
          return []


      knn = NearestNeighbors(n_neighbors=self.k + 1, algorithm="auto")
      knn.fit(all_features)
      _, indices = knn.kneighbors(minority_features)


      neighbor_ratios = []
      for idx_list in indices:
          neighbor_labels = all_labels[idx_list[1:]]
          majority_neighbors = np.sum(neighbor_labels != minority_label)
          ratio = majority_neighbors / self.k
          neighbor_ratios.append(ratio)


      valid = [(i, r) for i, r in enumerate(neighbor_ratios) if r < 1.0]
      if not valid:
          return []


      ratio_sum = sum(r for _, r in valid)


      # Uniform distribution if the class is well separated (ratio_sum == 0)
      if ratio_sum == 0:
          weights = [1.0 / len(valid)] * len(valid)
      else:
          weights = [r / ratio_sum for _, r in valid]


      # Base counts using int() to avoid rounding errors from round()
      counts_per_point = [(valid[i][0], int(w * samples_to_add)) for i, w in enumerate(weights)]


      # Distribute the remainder to the first points
      assigned = sum(c for _, c in counts_per_point)
      remainder = samples_to_add - assigned
      for i in range(remainder):
          idx, cnt = counts_per_point[i % len(counts_per_point)]
          counts_per_point[i % len(counts_per_point)] = (idx, cnt + 1)


      synthetic = []
      for sample_idx, n_samples in counts_per_point:
          if n_samples == 0:
              continue


          x = minority_features[sample_idx]
          neighbor_idx_list = indices[sample_idx][1:]
          neighbor_labels_list = all_labels[neighbor_idx_list]
          neighbor_features_list = all_features[neighbor_idx_list]


          minority_neighbor_idx = [
              j for j, lbl in enumerate(neighbor_labels_list)
              if lbl == minority_label
          ]


          # Fallback: if no minority neighbours, use all available neighbours
          if not minority_neighbor_idx:
              minority_neighbor_idx = list(range(len(neighbor_idx_list)))


          for _ in range(n_samples):
              chosen = random.choice(minority_neighbor_idx)
              neighbor_feat = neighbor_features_list[chosen]
              synthetic_feat = self._interpolate(x, neighbor_feat)
              synthetic.append((float(minority_label), Vectors.dense(synthetic_feat.tolist())))


      return synthetic



    # ------------------------------------------------------------------ #
    #  Equivalent of transform() in ADASYN.scala                         #
    # ------------------------------------------------------------------ #
    def fit_transform(self, df: DataFrame, sample_fraction: float = 0.1) -> DataFrame:
      spark = df.sparkSession


      # Encode strings -> numeric indices
      from pyspark.ml.feature import StringIndexer
      label_is_string = isinstance(df.schema[self.label_col].dataType, StringType)
      if label_is_string:
          indexer = StringIndexer(inputCol=self.label_col, outputCol="_label_idx")
          indexer_model = indexer.fit(df)
          df_indexed = indexer_model.transform(df) \
              .drop(self.label_col) \
              .withColumnRenamed("_label_idx", self.label_col)
          labels_list = indexer_model.labels
      else:
          df_indexed = df
          labels_list = None


      # --- STRATIFIED SAMPLING for KNN ---
      total_count = df_indexed.count()
      sample_size = max(int(total_count * sample_fraction), 10000)
      base_fraction = min(sample_size / total_count, 1.0)
      rare_threshold = 2000


      # Count on the full dataset (required before sampleBy)
      counts_rows = df_indexed.groupBy(self.label_col).count().orderBy(desc("count")).collect()
      counts = {row[self.label_col]: row["count"] for row in counts_rows}


      # Stratified fractions: rare classes at 100%, others proportionally
      fractions = {
          float(label): 1.0 if count < rare_threshold else base_fraction
          for label, count in counts.items()
      }
      df_sample = df_indexed.sampleBy(self.label_col, fractions=fractions, seed=42)


      print(f"Original dataset: {total_count} rows")
      print(f"Stratified sample: {df_sample.count()} rows")


      majority_label = counts_rows[0][self.label_col]
      majority_count = counts_rows[0]["count"]
      minority_labels = [lbl for lbl in counts if lbl != majority_label]


      # Run KNN on the sample to save memory
      all_rows_sample = df_sample.select(self.label_col, self.features_col).collect()
      all_features_sample = np.array([row[self.features_col].toArray() for row in all_rows_sample])
      all_labels_sample = np.array([float(row[self.label_col]) for row in all_rows_sample])


      synthetic_dfs = []
      for label in minority_labels:
          n_to_add = self._samples_to_add(label, counts[label], majority_count)
          synthetic_rows = self._oversample(all_features_sample, all_labels_sample, float(label), n_to_add)
          if synthetic_rows:
              synthetic_df = spark.createDataFrame(synthetic_rows, self._get_schema(df_indexed))
              synthetic_dfs.append(synthetic_df)


      if self.oversamples_only:
          result = synthetic_dfs[0]
          for sdf in synthetic_dfs[1:]:
              result = result.union(sdf)
      else:
          base = df_indexed.select(
              col(self.label_col).cast(DoubleType()),
              col(self.features_col)
          )
          result = base
          for sdf in synthetic_dfs:
              result = result.union(sdf)


      if label_is_string and labels_list is not None:
          from pyspark.sql.functions import udf
          idx_to_str = udf(lambda i: labels_list[int(i)], StringType())
          result = result \
              .withColumn("_original_label", idx_to_str(col(self.label_col).cast("int"))) \
              .drop(self.label_col) \
              .withColumnRenamed("_original_label", self.label_col)


      return result


    def _get_schema(self, df):
          return StructType([
              StructField(self.label_col, DoubleType(), True),
              StructField(self.features_col, VectorUDT(), True)
          ])


In [ ]:
spark = SparkSession.builder.appName("ADASYN").getOrCreate()


# Fit a lightweight preprocessing pipeline to encode the Confidence feature
preprocessing_pipeline = PrePipeline(stages=[confidence_indexer, confidence_ohe])
preprocessing_model = preprocessing_pipeline.fit(df_train)


# Apply preprocessing to df_train, adding the Confidence_ohe column
df_train = preprocessing_model.transform(df_train)


feature_cols = ["NDVI", "FDI", "NDWI", "NRD", "NDMI", "BSI", "CON", "DIS", "ENER", "COR", "PC_1", "Confidence_ohe"]


assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
df_assembled = assembler.transform(df_train).select("Class", "features")


# Oversample minority classes using ADASYN, targeting 10k samples per class
adasyn = ADASYN(
    k=5,
    label_col="Class",
    features_col="features",
    target=10000,
    oversamples_only=False
)


# Fit and transform using a 5% sample fraction for neighbour estimation
df_train = adasyn.fit_transform(df_assembled, sample_fraction=0.05)
df_train.groupBy("Class").count().show()


In [ ]:

# Same feature names and order as defined in the VectorAssembler
feature_names = ["NDVI", "FDI", "NDWI", "NRD", "NDMI", "BSI",
                 "CON", "DIS", "ENER", "COR", "PC_1"]


# Append OHE column names for Confidence (one per category, e.g. 3: High/Medium/Low)
n_confidence_cats = 3  # adjust to the actual number of categories
ohe_names = [f"Confidence_ohe_{i}" for i in range(n_confidence_cats)]


all_feature_names = feature_names + ohe_names


# Explode the features vector into individual columns
df_to_save = df_train.withColumn("_arr", vector_to_array(col("features")))


for i, name in enumerate(all_feature_names):
    df_to_save = df_to_save.withColumn(name, col("_arr")[i])


# Drop intermediate vector columns before saving
df_to_save = df_to_save.drop("features", "_arr")


df_to_save.write.csv(
    "/content/drive/MyDrive/DDAM project/data/df_train_OVERSAMPLING.csv",
    header=True,
    mode="overwrite"
)


## Random search Cross Validation



In [ ]:
feature_cols = [
    "NDVI", "FDI", "NDWI", "NRD", "NDMI", "BSI", "CON", "DIS", "ENER", "COR", "PC_1", "Confidence_ohe"
]


label = "Class"


# Indexes the target column into numerical labels
indexer = StringIndexer(inputCol=label, outputCol="label")


# Assembles the features into a single column vector
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")


# Defines the XGBoost classifier
xgb_estimator = SparkXGBClassifier(
    features_col="features",
    label_col="label",
    num_workers=2,
    seed=42
)


In [ ]:
evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1"
)


pipeline = Pipeline(stages=[confidence_indexer, confidence_ohe, indexer, assembler, xgb_estimator])


# Defines a grid larger than NUM_SAMPLES to enable random sampling
large_param_grid = (
    ParamGridBuilder()
    .addGrid(xgb_estimator.max_depth, [4, 6, 8])        # 3 options
    .addGrid(xgb_estimator.subsample, [0.7, 0.8, 1.0])  # 3 options
    .addGrid(xgb_estimator.n_estimators, [50, 100])     # 2 options
    .build()  # Total: 3 * 3 * 2 = 18 combinations
)


NUM_SAMPLES = 8
SEED = 42


# Randomly samples NUM_SAMPLES combinations from the full grid
sampled_paramGrid = random.sample(large_param_grid, NUM_SAMPLES)


crossval = CrossValidator(
    estimator=pipeline,
    estimatorParamMaps=sampled_paramGrid,
    evaluator=evaluator,
    numFolds=5,
    seed=SEED
)


print(f"\n Starting Random Search Cross-Validation ({NUM_SAMPLES} combinations)...")


cvModel = crossval.fit(df_train)


# Extract the best model and the average CV metrics
bestModel = cvModel.bestModel
avgMetrics = cvModel.avgMetrics


print(f"\n Cross-Validation completed. Best F1: {max(avgMetrics):.4f}")


In [ ]:
OUTPUT_PATH = "/content/drive/MyDrive/DDAM project/models/best_xgb_binary_OVERSAMPLING_model"

# Save on Drive
print(f"Salvataggio del modello in corso su: {OUTPUT_PATH} ...")
bestModel.write().overwrite().save(OUTPUT_PATH)

print("Modello salvato con successo!")


In [ ]:
# Apply the same preprocessing pipeline used before undersampling
df_test_pre = preprocessing_model.transform(df_test)  # adds Confidence_indexed, Confidence_ohe


# Assemble features using the same columns as in training
feature_cols = ["NDVI", "FDI", "NDWI", "NRD", "NDMI", "BSI",
                "CON", "DIS", "ENER", "COR", "PC_1", "Confidence_ohe"]
assembler_test = VectorAssembler(inputCols=feature_cols, outputCol="features")


df_test = assembler_test.transform(df_test_pre).select("features", "Class")


# Apply the best model to the test set
predictions = bestModel.transform(df_test)
predictions.cache()
predictions.count()  # Forces materialization (cache is lazy)


predictions.select(col("label"), col("prediction")).show(5)


metrics_to_evaluate = ["accuracy", "f1", "weightedPrecision", "weightedRecall"]


print("\n--- Detailed Performance Metrics (Test Set) ---")


results = {}
for metric_name in metrics_to_evaluate:
    test_evaluator = MulticlassClassificationEvaluator(
        labelCol="label",
        predictionCol="prediction",
        metricName=metric_name
    )
    metric_value = test_evaluator.evaluate(predictions)
    results[metric_name] = metric_value
    print(f" {metric_name.ljust(20)}: {metric_value:.4f}")


# Confusion matrix — essential for per-class precision/recall analysis
print("\n--- Confusion Matrix ---")


confusion_matrix = predictions.groupBy('label').pivot('prediction').count().fillna(0).orderBy('label')
confusion_matrix.show()


In [ ]:
from pyspark.ml.feature import IndexToString
from pyspark.mllib.evaluation import MulticlassMetrics


# Converts the numerical prediction indices back to the original string labels
labelConverter = IndexToString(
    inputCol="prediction",
    outputCol="predictedLabel",
    labels=bestModel.stages[2].labels
)


predictions_with_strings = labelConverter.transform(predictions)
predictions_with_strings.cache()
predictions_with_strings.count()


# Actual count per class
support_counts_df = predictions_with_strings.groupBy("label").count()
support_map = support_counts_df.rdd.collectAsMap()


# MulticlassMetrics requires an RDD of (prediction, label) pairs as Double
prediction_and_labels = (
    predictions_with_strings.select(col("prediction"), col("label"))
    .rdd.map(lambda row: tuple(map(float, row)))
)


metrics = MulticlassMetrics(prediction_and_labels)
labels = predictions_with_strings.select("label").distinct().rdd.flatMap(lambda x: x).collect()


print(f"{'Classe':<25} {'Support':<10} {'Precision':<15} {'Recall':<15} {'F1-Score':<15}")
print("-" * 80)


for label_index in sorted(labels):
    class_precision = metrics.precision(label=label_index)
    class_recall    = metrics.recall(label=label_index)
    class_f1        = metrics.fMeasure(label=label_index)
    class_support   = support_map.get(label_index, 0)
    class_string    = bestModel.stages[2].labels[int(label_index)]


    print(f"{class_string:<25.25} {class_support:<10} {class_precision:<15.4f} {class_recall:<15.4f} {class_f1:<15.4f}")


print("-" * 80)


predictions_with_strings.unpersist()
predictions.unpersist()


In [ ]:
# ── SAVE ADASYN Multi ─────────────────────────────────────────────────────────
_labels_adasyn_multi = bestModel.stages[0].labels
_md_idx_ad = float(_labels_adasyn_multi.index("Marine Debris"))
saved_adasyn_multi = {
    "config":    "ADASYN Multi",
    "precision": round(metrics.precision(label=_md_idx_ad), 4),
    "recall":    round(metrics.recall(label=_md_idx_ad),    4),
    "f1":        round(metrics.fMeasure(label=_md_idx_ad),  4),
}
print("Saved ADASYN Multi:", saved_adasyn_multi)


# **Oversampling Variabile Target Binaria**

In [ ]:
# !pip install pyspark
from pyspark.sql import SparkSession
ss = SparkSession.builder \
    .master("local[*]") \
    .appName("imbalanced") \
    .getOrCreate()


In [ ]:
path_train = "/content/drive/MyDrive/DDAM project/data/train_test/train_set.parquet"
path_test  = "/content/drive/MyDrive/DDAM project/data/train_test/test_set.parquet"

df_train = ss.read.parquet(path_train)
df_test = ss.read.parquet(path_test)

In [ ]:
df_train.show()

In [ ]:
df_train.dtypes

In [ ]:
from pyspark.sql import functions as F

# Marine Debris = 1, Others = 0
df_train = df_train.withColumn(
  'Class',
  F.when(F.col('Class') == 'Marine Debris', 1).otherwise(0)
)

df_test = df_test.withColumn(
  'Class',
  F.when(F.col('Class') == 'Marine Debris', 1).otherwise(0)
)

## Drop columns

In [ ]:
columns_to_drop = ["XCoords", "YCoords", "Image", "Tile", "Date"]

df_train = df_train.drop(*columns_to_drop)
df_test = df_test.drop(*columns_to_drop)

In [ ]:
df_train.show()

## Confidence Conversion

In [ ]:
# Converts Confidence (string) to a numeric index
confidence_indexer = StringIndexer(
    inputCol="Confidence",
    outputCol="Confidence_indexed",
    handleInvalid="keep"
)

# One-Hot Encoding of the Confidence index
confidence_ohe = OneHotEncoder(
    inputCol="Confidence_indexed",
    outputCol="Confidence_ohe",
    dropLast=False  # Retains all categories
)

# **ADASYN**

In [ ]:
spark = SparkSession.builder.appName("ADASYN").getOrCreate()


# Fit a lightweight preprocessing pipeline to encode the Confidence feature
preprocessing_pipeline = PrePipeline(stages=[confidence_indexer, confidence_ohe])
preprocessing_model = preprocessing_pipeline.fit(df_train)


# Apply preprocessing to df_train, adding the Confidence_ohe column
df_train = preprocessing_model.transform(df_train)


feature_cols = ["NDVI", "FDI", "NDWI", "NRD", "NDMI", "BSI", "CON", "DIS", "ENER", "COR", "PC_1", "Confidence_ohe"]


assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
df_assembled = assembler.transform(df_train).select("Class", "features")


# Oversample minority classes using ADASYN, targeting 10k samples per class
adasyn = ADASYN(
    k=5,
    label_col="Class",
    features_col="features",
    target=10000,
    oversamples_only=False
)


# Fit and transform using a 5% sample fraction for neighbour estimation
df_train = adasyn.fit_transform(df_assembled, sample_fraction=0.05)
df_train.groupBy("Class").count().show()


In [ ]:

# Same feature names and order as defined in the VectorAssembler
feature_names = ["NDVI", "FDI", "NDWI", "NRD", "NDMI", "BSI",
                 "CON", "DIS", "ENER", "COR", "PC_1"]


# Append OHE column names for Confidence (one per category, e.g. 3: High/Medium/Low)
n_confidence_cats = 3  # adjust to the actual number of categories
ohe_names = [f"Confidence_ohe_{i}" for i in range(n_confidence_cats)]


all_feature_names = feature_names + ohe_names


# Explode the features vector into individual columns
df_to_save = df_train.withColumn("_arr", vector_to_array(col("features")))


for i, name in enumerate(all_feature_names):
    df_to_save = df_to_save.withColumn(name, col("_arr")[i])


# Drop intermediate vector columns before saving
df_to_save = df_to_save.drop("features", "_arr")


df_to_save.write.csv(
    "/content/drive/MyDrive/DDAM project/data/df_train_binary_OVERSAMPLING.csv",
    header=True,
    mode="overwrite"
)


## Random search Cross Validation



In [ ]:
feature_cols = [
    "NDVI", "FDI", "NDWI", "NRD", "NDMI", "BSI", "CON", "DIS", "ENER", "COR", "PC_1", "Confidence_ohe"
]


label = "Class"


# Indexes the target column into numerical labels
indexer = StringIndexer(inputCol=label, outputCol="label")


# Assembles the features into a single column vector
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")


# Defines the XGBoost classifier
xgb_estimator = SparkXGBClassifier(
    features_col="features",
    label_col="label",
    num_workers=2,
    seed=42
)


In [ ]:
evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1"
)


pipeline = Pipeline(stages=[confidence_indexer, confidence_ohe, indexer, assembler, xgb_estimator])


# Defines a grid larger than NUM_SAMPLES to enable random sampling
large_param_grid = (
    ParamGridBuilder()
    .addGrid(xgb_estimator.max_depth, [4, 6, 8])        # 3 options
    .addGrid(xgb_estimator.subsample, [0.7, 0.8, 1.0])  # 3 options
    .addGrid(xgb_estimator.n_estimators, [50, 100])     # 2 options
    .build()  # Total: 3 * 3 * 2 = 18 combinations
)


NUM_SAMPLES = 8
SEED = 42


# Randomly samples NUM_SAMPLES combinations from the full grid
sampled_paramGrid = random.sample(large_param_grid, NUM_SAMPLES)


crossval = CrossValidator(
    estimator=pipeline,
    estimatorParamMaps=sampled_paramGrid,
    evaluator=evaluator,
    numFolds=5,
    seed=SEED
)


print(f"\n Starting Random Search Cross-Validation ({NUM_SAMPLES} combinations)...")


cvModel = crossval.fit(df_train)


# Extract the best model and the average CV metrics
bestModel = cvModel.bestModel
avgMetrics = cvModel.avgMetrics


print(f"\n Cross-Validation completed. Best F1: {max(avgMetrics):.4f}")


In [ ]:
OUTPUT_PATH = "/content/drive/MyDrive/DDAM project/models/best_xgb_binary_OVERSAMPLING_model"

# Save on Drive
print(f"Salvataggio del modello in corso su: {OUTPUT_PATH} ...")
bestModel.write().overwrite().save(OUTPUT_PATH)

print("Modello salvato con successo!")


In [ ]:
# --- Preprocessing & Feature Assembly ---
feature_cols = ["NDVI", "FDI", "NDWI", "NRD", "NDMI", "BSI",
                "CON", "DIS", "ENER", "COR", "PC_1", "Confidence_ohe"]

df_test_pre = preprocessing_model.transform(df_test)
df_test_assembled = VectorAssembler(inputCols=feature_cols, outputCol="features") \
                        .transform(df_test_pre) \
                        .select("features", col("Class").cast("double"))  # ← FIX: allinea il tipo a quello del train
# --- Predizioni ---
predictions = bestModel.transform(df_test_assembled).cache()
predictions.count()  # materializza cache

# --- Metriche ---
for metric in ["accuracy", "f1", "weightedPrecision", "weightedRecall"]:
    val = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName=metric).evaluate(predictions)
    print(f"{metric:<20}: {val:.4f}")

auc = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC").evaluate(predictions)
print(f"{'AUC-ROC':<20}: {auc:.4f}")

# --- Confusion Matrix ---
print("\n--- Confusion Matrix ---")
predictions.groupBy("label").pivot("prediction").count().fillna(0).orderBy("label").show()


In [ ]:
from pyspark.ml.feature import IndexToString
from pyspark.mllib.evaluation import MulticlassMetrics


# Converts the numerical prediction indices back to the original string labels
labelConverter = IndexToString(
    inputCol="prediction",
    outputCol="predictedLabel",
    labels=bestModel.stages[2].labels
)


predictions_with_strings = labelConverter.transform(predictions)
predictions_with_strings.cache()
predictions_with_strings.count()


# Actual count per class -> dictionary for fast lookup in the loop
support_counts_df = predictions_with_strings.groupBy("label").count()
support_map = support_counts_df.rdd.collectAsMap()


# MulticlassMetrics requires an RDD of (prediction, label) pairs as Double
prediction_and_labels = (
    predictions_with_strings.select(col("prediction"), col("label"))
    .rdd.map(lambda row: tuple(map(float, row)))
)


metrics = MulticlassMetrics(prediction_and_labels)
labels = predictions_with_strings.select("label").distinct().rdd.flatMap(lambda x: x).collect()


print("\n--- Per-Class Metrics (Test Set) ---")
print(f"{'Class':<25} {'Support':<10} {'Precision':<15} {'Recall':<15} {'F1-Score':<15}")
print("-" * 80)


for label_index in sorted(labels):
    class_precision = metrics.precision(label=label_index)
    class_recall    = metrics.recall(label=label_index)
    class_f1        = metrics.fMeasure(label=label_index)
    class_support   = support_map.get(label_index, 0)
    class_string    = bestModel.stages[2].labels[int(label_index)]


    print(f"{class_string:<25.25} {class_support:<10} {class_precision:<15.4f} {class_recall:<15.4f} {class_f1:<15.4f}")


print("-" * 80)


predictions_with_strings.unpersist()
predictions.unpersist()


In [ ]:
# ── SAVE ADASYN Binary ────────────────────────────────────────────────────────
saved_adasyn_binary = {
    "config":    "ADASYN Binary",
    "precision": round(metrics.precision(label=1.0), 4),
    "recall":    round(metrics.recall(label=1.0),    4),
    "f1":        round(metrics.fMeasure(label=1.0),  4),
}
print("Saved ADASYN Binary:", saved_adasyn_binary)


# PLOT

In [ ]:
import plotly.graph_objects as go

# ── Baseline da Notebook 1 (hardcoded, non cambiano) ─────────────────────────
baseline = [
    {"config": "Normal Multi",  "precision": 0.8728, "recall": 0.8900, "f1": 0.8813},
    {"config": "Normal Binary", "precision": 0.8954, "recall": 0.8998, "f1": 0.8976},
]

# ── Risultati dinamici da questo notebook ─────────────────────────────────────
dynamic = [saved_cc_multi, saved_adasyn_multi, saved_cc_binary, saved_adasyn_binary]

all_configs = baseline + dynamic

labels = [m["config"]    for m in all_configs]
prec   = [m["precision"] for m in all_configs]
rec    = [m["recall"]    for m in all_configs]
f1s    = [m["f1"]        for m in all_configs]

# ── Plot ──────────────────────────────────────────────────────────────────────
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=rec, y=prec,
    mode="markers+text",
    text=labels,
    textposition="top center",
    marker=dict(
        size=[v * 30 for v in f1s],
        color=f1s,
        colorscale="RdYlGn",
        showscale=True,
        colorbar=dict(title="F1-Score"),
        line=dict(width=1, color="white"),
    ),
    hovertemplate=(
        "<b>%{text}</b><br>"
        "Recall: %{x:.4f}<br>"
        "Precision: %{y:.4f}<br>"
        "F1: %{marker.color:.4f}<br>"
        "<extra></extra>"
    ),
    cliponaxis=False,
))

# Linea diagonale P = R
fig.add_shape(type="line", x0=0.65, y0=0.65, x1=1.0, y1=1.0,
              line=dict(color="gray", dash="dot", width=1))

fig.update_layout(
    title="Marine Debris — Precision vs Recall Trade-off",
    xaxis=dict(title="Recall",    range=[0.65, 1.05], dtick=0.05),
    yaxis=dict(title="Precision", range=[0.35, 1.00], dtick=0.10),
)

fig.show()
# fig.write_image("prec_rec_tradeoff.png")  # decommentare per export
